<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v1/blob/master/Final_Results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final_Results — Held-Out Test Set Evaluation

**⚠️ READ BEFORE RUNNING**

This notebook evaluates all key models on the **held-out test set** (1,500 samples, never seen during training or validation).

**Models evaluated:**
- Baselines: MobileNetV2, MobileNetV3 (seed 74)
- Teachers: VGG_Pretrained, ResNet_Pretrained (seed 85)
- KD: `vgg_mv2_ft`, `vgg_mv2_scratch`, `vgg_mv3_scratch` (best per student)
- Unstructured pruning: V2 + V3 at 10 / 20 / 30%
- Structured pruning: V3 3% (only structured checkpoint that passed NSE gate)

**STM32 on-device results** are loaded from `quantz_records.json` and shown alongside val/test accuracy wherever available.

In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place utils/ at: My Drive/stm32-thesis/utils/")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

from utils.dataset import prepare_dataset, get_test_loader
from utils.models  import (
    VWW_MobileNetV2, VWW_MobileNetV3,
    VGG_Pretrained, ResNet_Pretrained,
)
from utils.train import setup_device

device = setup_device(seed=41)
CKPT   = Path("/content/drive/My Drive/stm32-thesis/checkpoints")
QUANTZ_RECORDS_JSON = str(CKPT / "quantz_records.json")

print(f"Checkpoint dir : {CKPT}")
print(f"Dir exists     : {CKPT.exists()}")

In [ ]:
# ── Test set ─────────────────────────────────────────────────────────
prepare_dataset()
test_loader = get_test_loader(batch_size=64)
print("✅ Test loader ready")

In [ ]:
# ── Evaluation helpers ───────────────────────────────────────────────

def evaluate_model(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    correct = total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y  = X.to(device), y.to(device)
            preds = model(X).argmax(dim=1)
            correct += (preds == y).sum().item()
            total   += y.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(y.cpu().tolist())
    return correct / total, all_preds, all_labels


def load_and_eval(model_cls, ckpt_name, label=None):
    ckpt = CKPT / ckpt_name
    if not ckpt.exists():
        print(f"  ❌  {ckpt_name}  — NOT FOUND")
        return None, None, None
    model = model_cls().to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    acc, preds, labels = evaluate_model(model, test_loader)
    name = label or ckpt_name
    print(f"  ✅  {name:<50}  test: {acc*100:.2f}%")
    return acc, preds, labels


def plot_confusion(preds, labels, title, classes=("non_person", "person")):
    cm  = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title, fontsize=10)
    plt.tight_layout(); plt.show()


print("✅ Helpers ready")

In [ ]:
# ── Load STM32 results from quantz_records.json ─────────────────────
stm32 = {}   # name → {stm32_fp32_acc_%, stm32_int8_acc_%}

if os.path.exists(QUANTZ_RECORDS_JSON):
    with open(QUANTZ_RECORDS_JSON) as f:
        qrecords = json.load(f)
    for r in qrecords:
        stm32[r["name"]] = {
            "fp32_val"   : r.get("fp32_val_acc_%"),
            "stm32_fp32" : r.get("stm32_fp32_acc_%"),
            "stm32_int8" : r.get("stm32_int8_acc_%"),
            "nse"        : r.get("NSE"),
            "source"     : r.get("source"),
        }
    print(f"✅ Loaded STM32 records for {len(stm32)} models")
    for name, v in stm32.items():
        fp32_s = f"{v['stm32_fp32']:.2f}%" if v["stm32_fp32"] else "pending"
        int8_s = f"{v['stm32_int8']:.2f}%" if v["stm32_int8"] else "pending"
        print(f"  {name:<42}  FP32: {fp32_s:>10}  INT8: {int8_s:>10}")
else:
    print(f"⚠️  quantz_records.json not found — STM32 columns will be empty")
    print(f"   Expected at: {QUANTZ_RECORDS_JSON}")

In [ ]:
# ── Evaluate all models on held-out test set ─────────────────────────
results = {}

# ── Baselines ────────────────────────────────────────────────────────
print("── Baselines ──")
acc, preds, labels = load_and_eval(VWW_MobileNetV2, "mobilenetv2_seed_74.pth",
                                   "MobileNetV2 baseline")
results["MobileNetV2 baseline"] = {"acc": acc, "preds": preds, "labels": labels,
                                   "stm32_key": "mobilenetv2_seed_74"}

acc, preds, labels = load_and_eval(VWW_MobileNetV3, "mobilenetv3_seed_74.pth",
                                   "MobileNetV3 baseline")
results["MobileNetV3 baseline"] = {"acc": acc, "preds": preds, "labels": labels,
                                   "stm32_key": "mobilenetv3_seed_74"}

# ── Teachers ─────────────────────────────────────────────────────────
print("\n── Teachers ──")
acc, preds, labels = load_and_eval(VGG_Pretrained, "vgg_pretrained_seed_85.pth",
                                   "VGG Pretrained (teacher)")
results["VGG Pretrained (teacher)"] = {"acc": acc, "preds": preds, "labels": labels,
                                        "stm32_key": None}

acc, preds, labels = load_and_eval(ResNet_Pretrained, "resnet_pretrained_seed_85.pth",
                                   "ResNet Pretrained (teacher)")
results["ResNet Pretrained (teacher)"] = {"acc": acc, "preds": preds, "labels": labels,
                                           "stm32_key": None}

# ── KD — best per student ─────────────────────────────────────────────
print("\n── Knowledge Distillation ──")
acc, preds, labels = load_and_eval(VWW_MobileNetV2, "vgg_mv2_ft.pth",
                                   "MobileNetV2 KD VGG-FT")
results["MobileNetV2 KD VGG-FT"] = {"acc": acc, "preds": preds, "labels": labels,
                                     "stm32_key": "vgg_mv2_ft"}

acc, preds, labels = load_and_eval(VWW_MobileNetV2, "vgg_mv2_scratch.pth",
                                   "MobileNetV2 KD VGG-scratch")
results["MobileNetV2 KD VGG-scratch"] = {"acc": acc, "preds": preds, "labels": labels,
                                          "stm32_key": "vgg_mv2_scratch"}

acc, preds, labels = load_and_eval(VWW_MobileNetV3, "vgg_mv3_scratch.pth",
                                   "MobileNetV3 KD VGG-scratch")
results["MobileNetV3 KD VGG-scratch"] = {"acc": acc, "preds": preds, "labels": labels,
                                          "stm32_key": "vgg_mv3_scratch"}

# ── Unstructured pruning ──────────────────────────────────────────────
print("\n── Unstructured Pruning ──")
for arch, cls in [("mobilenetv2", VWW_MobileNetV2), ("mobilenetv3", VWW_MobileNetV3)]:
    for pct in [10, 20, 30]:
        ckpt_name = f"{arch}_unstructured_{pct}pct.pth"
        label     = f"{'MobileNetV2' if arch == 'mobilenetv2' else 'MobileNetV3'} unstructured {pct}%"
        stm32_key = f"{arch}_unstructured_{pct}pct"
        acc, preds, labels = load_and_eval(cls, ckpt_name, label)
        results[label] = {"acc": acc, "preds": preds, "labels": labels,
                          "stm32_key": stm32_key}

# ── Structured pruning — V3 3% only ──────────────────────────────────
print("\n── Structured Pruning ──")
acc, preds, labels = load_and_eval(VWW_MobileNetV3, "mobilenetv3_structured_3pct.pth",
                                   "MobileNetV3 structured 3%")
results["MobileNetV3 structured 3%"] = {"acc": acc, "preds": preds, "labels": labels,
                                         "stm32_key": "mobilenetv3_structured_3pct"}

print(f"\n✅ Evaluated {sum(1 for r in results.values() if r['acc'] is not None)}"
      f"/{len(results)} models successfully")

In [ ]:
# ── 95% confidence interval helper ──────────────────────────────────
def wilson_ci_95(acc_pct, n=1500):
    """Wilson score 95% CI. Returns (lo%, hi%) as floats."""
    p = acc_pct / 100
    z = 1.96
    center = (p + z**2 / (2*n)) / (1 + z**2 / n)
    margin = z * (p*(1-p)/n + z**2/(4*n**2))**0.5 / (1 + z**2/n)
    return round((center - margin)*100, 2), round((center + margin)*100, 2)

print("✅ wilson_ci_95 ready  (n=1500, 95% CI ≈ ±1.3%)")

In [ ]:
# ── Final results table ──────────────────────────────────────────────
# Val accs from training runs
VAL_ACCS = {
    "MobileNetV2 baseline"        : 78.40,
    "MobileNetV3 baseline"        : 79.13,
    "VGG Pretrained (teacher)"    : 89.07,
    "ResNet Pretrained (teacher)" : 87.93,
    "MobileNetV2 KD VGG-FT"      : 80.07,
    "MobileNetV2 KD VGG-scratch"  : 79.53,
    "MobileNetV3 KD VGG-scratch"  : 79.53,
    "MobileNetV2 unstructured 10%": 78.93,
    "MobileNetV2 unstructured 20%": 78.53,
    "MobileNetV2 unstructured 30%": 78.27,
    "MobileNetV3 unstructured 10%": 78.67,
    "MobileNetV3 unstructured 20%": 78.27,
    "MobileNetV3 unstructured 30%": 78.47,
    "MobileNetV3 structured 3%"   : 78.73,
}

rows = []
for name, r in results.items():
    if r["acc"] is None:
        continue
    val      = VAL_ACCS.get(name)
    test_acc = round(r["acc"] * 100, 2)
    delta    = round(test_acc - val, 2) if val else None
    ci_lo, ci_hi = wilson_ci_95(test_acc)

    s32      = stm32.get(r["stm32_key"], {}) if r["stm32_key"] else {}
    rows.append({
        "model"          : name,
        "val_acc_%"      : val,
        "test_acc_%"     : test_acc,
        "test_ci_95"     : f"[{ci_lo:.1f}, {ci_hi:.1f}]",
        "test_delta_%"   : delta,
        "stm32_fp32_%"   : s32.get("stm32_fp32"),
        "stm32_int8_%"   : s32.get("stm32_int8"),
        "NSE"            : s32.get("nse"),
    })

df = pd.DataFrame(rows)

W = 105
print("=" * W)
print(f"{'FINAL TEST SET RESULTS  (n=1500, held-out)':^{W}}")
print("=" * W)
print(f"{'Model':<42} {'Val':>7}  {'Test':>7}  {'Δ(T-V)':>7}  "
      f"{'STM32 FP32':>11}  {'STM32 INT8':>11}  {'NSE':>7}")
print("-" * W)

sections = [
    ("Baselines", ["MobileNetV2 baseline", "MobileNetV3 baseline"]),
    ("Teachers",  ["VGG Pretrained (teacher)", "ResNet Pretrained (teacher)"]),
    ("Knowledge Distillation",
                  ["MobileNetV2 KD VGG-FT", "MobileNetV2 KD VGG-scratch",
                   "MobileNetV3 KD VGG-scratch"]),
    ("Unstructured Pruning",
                  ["MobileNetV2 unstructured 10%", "MobileNetV2 unstructured 20%",
                   "MobileNetV2 unstructured 30%", "MobileNetV3 unstructured 10%",
                   "MobileNetV3 unstructured 20%", "MobileNetV3 unstructured 30%"]),
    ("Structured Pruning (3% boundary — all runs ≥5% fail NSE gate)", ["MobileNetV3 structured 3%"]),
]

for section_name, model_names in sections:
    print(f"  {section_name}")
    for name in model_names:
        row = df[df["model"] == name]
        if row.empty: continue
        r = row.iloc[0]

        val_s   = f"{r['val_acc_%']:.2f}%"   if r["val_acc_%"]   is not None else "   —"
        test_s  = f"{r['test_acc_%']:.2f}%"
        delta_s = f"{r['test_delta_%']:+.2f}%" if r["test_delta_%"] is not None else "   —"
        ci_s    = r.get("test_ci_95", "         —")
        fp32_s  = f"{r['stm32_fp32_%']:.2f}%" if r["stm32_fp32_%"] is not None else "  pending"
        int8_s  = f"{r['stm32_int8_%']:.2f}%" if r["stm32_int8_%"] is not None else "  pending"
        nse_s   = f"{r['NSE']:.4f}"           if r["NSE"]          is not None else "     —"

        print(f"    {name:<40} {val_s:>7}  {test_s:>7}  {ci_s:>14}  {delta_s:>7}  "
              f"{fp32_s:>11}  {int8_s:>11}  {nse_s:>7}")
    print()

print("=" * W)
print("Δ(T-V) = Test − Val  (negative = val was optimistic, positive = test generalised better)")

In [ ]:
# ── Per-class breakdown for key models ──────────────────────────────
key_models = [
    "MobileNetV2 baseline",
    "MobileNetV3 baseline",
    "MobileNetV2 KD VGG-FT",
    "MobileNetV3 KD VGG-scratch",
    "MobileNetV2 unstructured 30%",
    "MobileNetV3 unstructured 30%",
    "MobileNetV3 structured 3%",
]

for name in key_models:
    if name not in results or results[name]["acc"] is None:
        continue
    r = results[name]
    print(f"\n── {name} ──")
    print(classification_report(
        r["labels"], r["preds"],
        target_names=["non_person", "person"],
        digits=4,
    ))

In [ ]:
# ── Confusion matrices ───────────────────────────────────────────────
for name in key_models:
    if name not in results or results[name]["acc"] is None:
        continue
    r = results[name]
    plot_confusion(r["preds"], r["labels"], title=name)

In [ ]:
# ── Accuracy vs compression chart ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Test Accuracy vs Compression Level", fontsize=13, fontweight="bold")

for ax, arch in zip(axes, ["MobileNetV2", "MobileNetV3"]):
    baseline_name = f"{arch} baseline"
    baseline_acc  = results[baseline_name]["acc"] * 100 if results.get(baseline_name, {}).get("acc") else None

    amounts  = [10, 20, 30]
    test_acc = []
    for pct in amounts:
        name = f"{arch} unstructured {pct}%"
        r    = results.get(name, {})
        test_acc.append(r["acc"] * 100 if r.get("acc") else None)

    valid = [(a, t) for a, t in zip(amounts, test_acc) if t is not None]
    if valid:
        xs, ys = zip(*valid)
        ax.plot(xs, ys, "o-", color="steelblue", linewidth=2,
                label="Unstructured pruning")

    if baseline_acc:
        ax.axhline(baseline_acc, color="gray", linestyle="--",
                   linewidth=1.5, label=f"Baseline {baseline_acc:.2f}%")

    # KD point for V2
    if arch == "MobileNetV2":
        kd_acc = results.get("MobileNetV2 KD VGG-FT", {}).get("acc")
        if kd_acc:
            ax.axhline(kd_acc * 100, color="green", linestyle=":",
                       linewidth=1.5, label=f"KD VGG-FT {kd_acc*100:.2f}%")
    if arch == "MobileNetV3":
        kd_acc = results.get("MobileNetV3 KD VGG-scratch", {}).get("acc")
        if kd_acc:
            ax.axhline(kd_acc * 100, color="green", linestyle=":",
                       linewidth=1.5, label=f"KD VGG-scratch {kd_acc*100:.2f}%")

    ax.set_title(arch)
    ax.set_xlabel("Pruning amount (%)")
    ax.set_ylabel("Test accuracy (%)")
    ax.set_xticks([10, 20, 30])
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plot_path = "/content/drive/My Drive/stm32-thesis/outputs/final_accuracy_chart.png"
Path(plot_path).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"✅ Chart saved → {plot_path}")

In [ ]:
# ── Save results to JSON ─────────────────────────────────────────────
summary = {}
for name, r in results.items():
    if r["acc"] is None: continue
    s32     = stm32.get(r["stm32_key"], {}) if r["stm32_key"] else {}
    summary[name] = {
        "val_acc_%"     : VAL_ACCS.get(name),
        "test_acc_%"    : round(r["acc"] * 100, 2),
        "stm32_fp32_%"  : s32.get("stm32_fp32"),
        "stm32_int8_%"  : s32.get("stm32_int8"),
        "NSE"           : s32.get("nse"),
    }

out_path = Path("/content/drive/My Drive/stm32-thesis/outputs/final_test_results.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(summary, indent=2))

print(f"✅ Saved to: {out_path}")
print(json.dumps(summary, indent=2))